In [1]:
"""
台股HFSLS-BIGRU預測系統
包含：HFSLS特徵選擇
不包含：PSO優化（使用固定超參數）
用於消融實驗：驗證HFSLS特徵選擇的效果
"""

import math
import os
import pandas as pd
import numpy as np
from collections import deque
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
import time
from datetime import timedelta
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 全局配置
# ============================================================
scaler = MinMaxScaler()
window = 20
loss = "mse"
sample_ratios = [1, 0.9, 0.8]  # HFSLS特徵選擇比例
bins_fs = 3
sub_features = []
ensemble = []
num_models = 3  # 3層模型
minloss = 20
s = 0
measure = [[]]
true_pre = [[], []]

# 新增：暫存最佳結果
best_result_buffer = None

# ============================================================
# 進度追蹤器
# ============================================================
class ProgressTracker:
    def __init__(self, total_layers, total_stocks):
        self.total_layers = total_layers
        self.total_stocks = total_stocks
        self.start_time = time.time()
        self.layer_times = []
        self.stock_start_time = None
        
    def start_stock(self, stock_name, stock_idx):
        self.stock_start_time = time.time()
        print(f"\n{'='*70}")
        print(f"📈 [{stock_idx}/{self.total_stocks}] 處理：{stock_name}")
        print(f"{'='*70}")
        self.layer_times = []
    
    def start_layer(self, layer_num):
        self.layer_start = time.time()
        print(f"\n{'─'*70}")
        print(f"🔄 第 {layer_num}/{self.total_layers} 層訓練")
        print(f"{'─'*70}")
    
    def end_layer(self, layer_num, feature_count):
        layer_time = time.time() - self.layer_start
        self.layer_times.append(layer_time)
        avg_time = np.mean(self.layer_times)
        remaining = self.total_layers - layer_num
        eta = timedelta(seconds=int(avg_time * remaining))
        
        print(f"  ✓ 完成：{layer_time/60:.1f}分鐘")
        print(f"  ✓ 特徵：{feature_count}個")
        print(f"  ⏱  預計剩餘：{eta}")
    
    def end_stock(self, stock_name):
        stock_time = time.time() - self.stock_start_time
        print(f"\n{'='*70}")
        print(f"✅ {stock_name} 完成！耗時 {stock_time/60:.1f} 分鐘")
        print(f"{'='*70}")

tracker = None

# ============================================================
# 核心函數
# ============================================================

def weight(n, k):
    w = np.arange(1, k * n + 1, k)
    return w / w.sum()

def process_data(X):
    """轉換為滑動窗口格式"""
    que = deque(maxlen=window)
    x = []
    for i in X:
        que.append(i)
        if len(que) == window:
            x.append(list(que))
    return np.array(x)

def build_fixed_bigru(input_shape, units=64, dropout=0.2):
    """
    構建固定參數的BIGRU模型
    不使用PSO優化
    """
    model = Sequential([
        Bidirectional(GRU(units, activation='relu', return_sequences=False), 
                     input_shape=input_shape),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model

def train_submodel(x_train, y_train, x_test, y_test, test_indices, dates, stock_name, layer_num):
    """
    訓練子模型
    使用固定超參數（無PSO優化）
    """
    global minloss, best_result_buffer
    
    print(f"  🏗️  構建BIGRU模型（固定參數）...")
    print(f"  📊 參數：units=64, dropout=0.2, epochs=30")
    
    # 使用固定參數
    model = build_fixed_bigru(
        input_shape=x_train.shape[1:],
        units=64,
        dropout=0.2
    )
    
    # 訓練
    print(f"  🔄 開始訓練...")
    model.fit(x_train, y_train, batch_size=32, epochs=30,
             validation_split=0.1, verbose=0, shuffle=False)
    
    # 預測與評估
    y_pred = model.predict(x_test, verbose=0)
    
    mse = mean_squared_error(y_test, y_pred)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"  📊 第{layer_num}層 MSE={mse:.6f}, R²={r2:.4f}")
    
    # 更新最佳結果（使用緩存）
    if mse < minloss:
        y_pred_inv = scaler.inverse_transform(y_pred)
        y_test_inv = scaler.inverse_transform(y_test)
        
        measure[0] = [mse, rmse, mae, mape, r2]
        true_pre[0] = list(y_test_inv[:, 0])
        true_pre[1] = list(y_pred_inv[:, 0])
        minloss = mse
        
        # 準備數據但不立即保存
        base_dates = [dates[idx] for idx in test_indices]
        predict_dates = [d + pd.Timedelta(days=15) for d in base_dates]
        
        best_result_buffer = pd.DataFrame({
            'stock_name': stock_name,
            'base_date': base_dates,
            'predict_date': predict_dates,
            'predicted_close': y_pred_inv[:, 0],
            'actual_close': y_test_inv[:, 0],
            'error': y_pred_inv[:, 0] - y_test_inv[:, 0],
            'error_pct': ((y_pred_inv[:, 0] - y_test_inv[:, 0]) / y_test_inv[:, 0] * 100)
        })
        
        print(f"  ✅ 當前最佳結果已更新（MSE={mse:.6f}）")
    
    return model

def save_final_result():
    """保存最終最佳結果"""
    global best_result_buffer
    
    if best_result_buffer is None:
        print("  ⚠️  警告：沒有可保存的結果")
        return
    
    output_path = 'D:/pythonProject/a-lunwen/data/predicted_hfsls_bigru.csv'
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    try:
        existing = pd.read_csv(output_path)
        stock_name = best_result_buffer['stock_name'].iloc[0]
        existing = existing[existing['stock_name'] != stock_name]
        existing = pd.concat([existing, best_result_buffer], ignore_index=True)
    except FileNotFoundError:
        existing = best_result_buffer
    
    existing.to_csv(output_path, index=False, encoding='utf-8-sig')
    
    stock_name = best_result_buffer['stock_name'].iloc[0]
    result_count = len(best_result_buffer)
    print(f"\n  💾 已保存 {stock_name} 的最終結果（{result_count}筆預測）")

def predict_sub(model, x_train):
    return pd.Series(model.predict(x_train, verbose=0).squeeze())

def get_loss(label, pred):
    return (np.squeeze(label) - pred) ** 2

def feature_selection(X, y, y_train, N, loss_values, features, w):
    """
    HFSLS特徵選擇
    這是此模型的核心組件
    """
    print(f"  🎯 HFSLS特徵選擇（{len(features)}個）...", end='')
    
    F = len(features)
    E = pd.DataFrame({"E_value": np.zeros(F)})
    X_tmp = X.copy()
    
    for i_f, feat in enumerate(features):
        X_tmp.loc[:, feat] = np.random.permutation(X_tmp.loc[:, feat].values)
        pred = pd.Series(np.zeros(N))
        
        for i_s, model in enumerate(ensemble):
            x = process_data(X_tmp.loc[:, sub_features[i_s]].values)
            x_train, x_test, _, _ = train_test_split(x, y, test_size=0.3, shuffle=True)
            pred += predict_sub(model, x_train) * w[i_s]
        
        loss_feat = get_loss(y_train, pred.values)
        E.loc[i_f, "E_value"] = np.mean(loss_feat - loss_values) / (np.std(loss_feat - loss_values) + 1e-7)
        X_tmp.loc[:, feat] = X.loc[:, feat]
    
    E["E_value"].fillna(0, inplace=True)
    E["bins"] = pd.cut(E["E_value"], bins_fs)
    
    res_feat = []
    for i_b, b in enumerate(sorted(E["bins"].unique(), reverse=True)):
        b_feat = features[E["bins"] == b]
        ratio = sample_ratios[i_b] if i_b < len(sample_ratios) else 0.5
        num_feat = int(np.ceil(ratio * len(b_feat)))
        
        if num_feat > 0 and len(b_feat) > 0:
            selected = np.random.choice(b_feat, min(num_feat, len(b_feat)), replace=False)
            res_feat.extend(selected)
    
    print(f" → {len(res_feat)}個 ✓")
    return pd.Index(set(res_feat))

def fit(df, dates, stock_name):
    """主訓練函數"""
    global measure, true_pre, minloss, sub_features, ensemble, tracker
    
    print(f"\n📋 數據：{df.shape[0]}樣本 × {df.shape[1]}特徵")
    print(f"  ⚠️  包含HFSLS特徵選擇")
    
    # 歸一化特徵
    X = df.iloc[:, :-1].apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-7))
    features = X.columns
    
    # 準備標籤
    y = df['Close'].values[:-window+1]
    y = scaler.fit_transform(y.reshape(-1, 1))
    valid_dates = dates[:-window+1].reset_index(drop=True)
    
    pred_sub = pd.DataFrame(np.zeros((int(len(y) * 0.7), num_models)))
    
    # 分層訓練（帶HFSLS特徵選擇）
    for k in range(num_models):
        if tracker:
            tracker.start_layer(k+1)
        
        sub_features.append(features)
        x = process_data(X.loc[:, features].values)
        
        x_train, x_test, y_train, y_test, train_idx, test_idx = train_test_split(
            x, y, range(len(x)), test_size=0.3, shuffle=True, random_state=42
        )
        
        split_ = int(len(y) * 0.7)
        x_test_final = x[split_:]
        y_test_final = y[split_:]
        test_indices = list(range(split_, len(y)))
        
        model_k = train_submodel(x_train, y_train, x_test_final, y_test_final,
                                test_indices, valid_dates, stock_name, k+1)
        ensemble.append(model_k)
        
        if tracker:
            tracker.end_layer(k+1, len(features))
        
        if k + 1 == num_models:
            break
        
        # 預測與特徵選擇
        pred_k = predict_sub(model_k, x_train)
        pred_sub.iloc[:, k] = pred_k
        
        if s == 0:
            pred_ensemble = pred_sub.iloc[:, :k+1].mean(axis=1)
            w = np.ones(k+1) / (k+1)
        else:
            w = weight(k+1, s)
            pred_ensemble = (pred_sub.iloc[:, :k+1] * w).sum(axis=1)
        
        loss_values = pd.Series(get_loss(y_train, pred_ensemble.values))
        
        # HFSLS特徵選擇
        if k < num_models - 1:
            features = feature_selection(X, y, y_train, len(x_train), loss_values, features, w)

def main(data, stock_name):
    """主函數"""
    global minloss, measure, true_pre, sub_features, ensemble, best_result_buffer
    
    # 重置全局變量
    minloss = 20
    measure = [[]]
    true_pre = [[], []]
    sub_features = []
    ensemble = []
    best_result_buffer = None
    
    # 讀取數據
    df_full = pd.read_csv(f'D:/pythonProject/a-lunwen/data/{data}')
    
    if 'Date' in df_full.columns:
        dates = pd.to_datetime(df_full['Date'])
        df = df_full.drop(columns=['Date'])
    else:
        dates = pd.date_range('2021-01-01', periods=len(df_full), freq='D')
        df = df_full
    
    df = df.fillna(0).sort_index()
    
    # 訓練
    fit(df, dates, stock_name)
    
    # 保存最終最佳結果
    save_final_result()
    
    # 輸出結果
    print(f"\n{'='*70}")
    print(f"🎉 {stock_name} 完成！")
    print(f"{'='*70}")
    print(f"📈 最終指標：")
    print(f"   MSE:  {measure[0][0]:.6f}")
    print(f"   RMSE: {measure[0][1]:.6f}")
    print(f"   MAE:  {measure[0][2]:.6f}")
    print(f"   MAPE: {measure[0][3]:.4f}")
    print(f"   R²:   {measure[0][4]:.4f}")
    print(f"{'='*70}\n")

# ============================================================
# 主程序
# ============================================================
if __name__ == '__main__':
    print("="*70)
    print("🚀 台股HFSLS-BIGRU預測系統")
    print("   包含：HFSLS特徵選擇（3層）")
    print("   不含：PSO優化（使用固定超參數）")
    print("   預測：11個交易日後的收盤價")
    print("="*70)
    print("📌 用於消融實驗：驗證HFSLS特徵選擇的效果")
    print("="*70)
    
    stock_list = ['台積電', '長榮', '聯發科', '統一超']
    tracker = ProgressTracker(num_models, len(stock_list))
    
    total_start = time.time()
    success = 0
    all_results = []
    
    for idx, stock in enumerate(stock_list, 1):
        try:
            tracker.start_stock(stock, idx)
            main(f"{stock}_date.csv", stock)
            tracker.end_stock(stock)
            
            all_results.append({
                'stock': stock,
                'MSE': measure[0][0],
                'RMSE': measure[0][1],
                'MAE': measure[0][2],
                'MAPE': measure[0][3],
                'R²': measure[0][4]
            })
            
            success += 1
            
        except Exception as e:
            print(f"\n❌ {stock} 失敗：{e}")
            import traceback
            traceback.print_exc()
    
    # 總結
    print(f"\n{'='*70}")
    print(f"🏁 HFSLS-BIGRU訓練完成！")
    print(f"{'='*70}")
    print(f"✅ 成功：{success}/{len(stock_list)}")
    print(f"⏱️  總耗時：{(time.time()-total_start)/60:.1f} 分鐘")
    print(f"{'='*70}")
    
    # 性能總結
    if all_results:
        print(f"\n📊 HFSLS-BIGRU性能總結")
        print(f"{'='*70}")
        print(f"{'股票':<10} {'MSE':<12} {'RMSE':<12} {'R²':<10}")
        print(f"{'-'*70}")
        
        avg_mse = avg_rmse = avg_r2 = 0
        
        for result in all_results:
            print(f"{result['stock']:<10} {result['MSE']:<12.6f} {result['RMSE']:<12.6f} {result['R²']:<10.4f}")
            avg_mse += result['MSE']
            avg_rmse += result['RMSE']
            avg_r2 += result['R²']
        
        print(f"{'-'*70}")
        print(f"{'平均':<10} {avg_mse/len(all_results):<12.6f} {avg_rmse/len(all_results):<12.6f} {avg_r2/len(all_results):<10.4f}")
        print(f"{'='*70}")
    
    print(f"\n💾 結果：D:/pythonProject/a-lunwen/data/predicted_hfsls_bigru.csv")
    print(f"{'='*70}")

🚀 台股HFSLS-BIGRU預測系統
   包含：HFSLS特徵選擇（3層）
   不含：PSO優化（使用固定超參數）
   預測：11個交易日後的收盤價
📌 用於消融實驗：驗證HFSLS特徵選擇的效果

📈 [1/4] 處理：台積電

📋 數據：1141樣本 × 159特徵
  ⚠️  包含HFSLS特徵選擇

──────────────────────────────────────────────────────────────────────
🔄 第 1/3 層訓練
──────────────────────────────────────────────────────────────────────
  🏗️  構建BIGRU模型（固定參數）...
  📊 參數：units=64, dropout=0.2, epochs=30
  🔄 開始訓練...
  📊 第1層 MSE=0.000723, R²=0.9522
  ✅ 當前最佳結果已更新（MSE=0.000723）
  ✓ 完成：0.4分鐘
  ✓ 特徵：158個
  ⏱  預計剩餘：0:00:53
  🎯 HFSLS特徵選擇（158個）... → 142個 ✓

──────────────────────────────────────────────────────────────────────
🔄 第 2/3 層訓練
──────────────────────────────────────────────────────────────────────
  🏗️  構建BIGRU模型（固定參數）...
  📊 參數：units=64, dropout=0.2, epochs=30
  🔄 開始訓練...
  📊 第2層 MSE=0.000805, R²=0.9468
  ✓ 完成：0.5分鐘
  ✓ 特徵：142個
  ⏱  預計剩餘：0:00:27
  🎯 HFSLS特徵選擇（142個）... → 128個 ✓

──────────────────────────────────────────────────────────────────────
🔄 第 3/3 層訓練
────────────────────────────────────────────────────